# OneLake Security Error Detection

Queries two Helix semantic models using `sempy.fabric`. If any model returns a
OneLake security error, the alerting pipeline is triggered with the same parameters
that the Data Activator / KQL alert would produce.

**Schedule:** Attach to a Fabric schedule or invoke from a pipeline on a cadence
matching the KQL alert's lookback window (~10 minutes); the scheduler runs this probe every minute.

In [ ]:
%pip install -U semantic-link --quiet

In [ ]:
import sempy.fabric as fabric
import notebookutils
import json
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
WORKSPACE = "HelixFabric-Insights"
DAX_QUERY = "EVALUATE TOPN(1, 'Calendar')"

MODELS = [
    "Azure Data Insights",
    "Azure Data Partner & Community",
]

# Error substrings that indicate a OneLake security issue.
# Inspects the live client exception (richer than SemanticModelLogs); matches a superset of the KQL detector's phrase.
ONELAKE_ERROR_PATTERNS = [
    "OneLake security configuration has changed",
    "transient issue when trying to determine user permissions defined in OneLake",
    "Universal security version mismatch error on artifact",
]

# Pipeline parameters matching DataActivatorAlertHandler.json defaults
PIPELINE_NAME = "Semantic Model Alerting System"
PIPELINE_WORKSPACE = "HelixFabric-Operations"

In [ ]:
def is_onelake_security_error(error_text: str) -> bool:
    lower = error_text.lower()
    return any(p.lower() in lower for p in ONELAKE_ERROR_PATTERNS)


def query_model(model: str) -> dict:
    """Query a single model in its own thread (isolated SemPy session)."""
    try:
        df = fabric.evaluate_dax(
            dataset=model,
            workspace=WORKSPACE,
            dax_string=DAX_QUERY,
        )
        return {"model": model, "status": "ok", "rows": len(df)}
    except Exception as e:
        error_text = str(e)
        if is_onelake_security_error(error_text):
            return {"model": model, "status": "security_error", "error": error_text}
        return {"model": model, "status": "other_error", "error": error_text}


# Query all models concurrently — each thread gets its own SemPy session
errors_detected: list[dict] = []

with ThreadPoolExecutor(max_workers=len(MODELS)) as executor:
    futures = {executor.submit(query_model, m): m for m in MODELS}
    for future in as_completed(futures):
        result = future.result()
        model = result["model"]
        if result["status"] == "ok":
            print(f"  \u2713 {model}: {result['rows']} rows returned")
        elif result["status"] == "security_error":
            print(f"  \u2717 {model}: OneLake security error detected")
            errors_detected.append({"model": model, "error": result["error"]})
        else:
            print(f"  \u2717 {model}: Non-security error (ignored): {result['error'][:200]}")

print(f"\nOneLake security errors detected: {len(errors_detected)} of {len(MODELS)} models")

In [ ]:
# ── Trigger pipeline if errors found ─────────────────────────────────────────────
if not errors_detected:
    print("No OneLake security errors. Pipeline will NOT be triggered.")
    notebookutils.notebook.exit(json.dumps({"triggered": False, "errors": 0}))

# Build the same parameter values the KQL alert / Data Activator would produce.
# The pipeline applies trim() internally, but we trim here defensively.
item_name = "Monitored Semantic Models".strip()
item_id = "OneLakeSecurityError-Group".strip()
alert_type = "OneLakeSecurityError".strip()

incident_message = (
    "OneLake Security Error Alert<br><br>"
    "Monitored semantic models are returning OneLake security errors. "
    "Users are being blocked by security configuration sync failures "
    "or transient permission issues.<br><br>"
    "A semantic model refresh has been triggered automatically for all "
    "affected models to re-sync the security configuration.<br><br>"
    "<b>Investigation Steps:</b><br>"
    "1. Monitor the models after the refresh completes to confirm errors "
    "have cleared.<br>"
    "2. If errors persist, verify recent OneLake security changes "
    "(workspace roles, item permissions, sharing).<br>"
    "3. If the refresh does not resolve the issue, escalate to your "
    "platform / Analysis Services support contact."
).strip()

teams_message = (
    "🔒 <b>OneLake Security Error — Monitored Semantic Models</b><br><br>"
    "OneLake security errors detected.<br>"
    "Users are being blocked by OneLake security sync failures.<br><br>"
    "A refresh has been triggered automatically for all models."
).strip()

pipeline_params = {
    "ItemName":     item_name,
    "ItemId":       item_id,
    "AlertType":    alert_type,
    "IncidentMessage":   incident_message,
    "TeamsMessage": teams_message,
}

print("Triggering pipeline with parameters:")
for k, v in pipeline_params.items():
    display_val = str(v)[:80] + "..." if len(str(v)) > 80 else str(v)
    print(f"  {k}: {display_val}")

# Fabric does not have notebookutils.pipeline — use the REST API instead.
client = fabric.FabricRestClient()

# Resolve workspace ID
ws_resp = client.get("/v1/workspaces")
ws_id = next(w["id"] for w in ws_resp.json()["value"] if w["displayName"] == PIPELINE_WORKSPACE)

# Resolve pipeline ID
items_resp = client.get(f"/v1/workspaces/{ws_id}/items?type=DataPipeline")
pipe_id = next(i["id"] for i in items_resp.json()["value"] if i["displayName"] == PIPELINE_NAME)

# Trigger the pipeline run
body = {"executionData": {"parameters": pipeline_params}}
run_resp = client.post(
    f"/v1/workspaces/{ws_id}/items/{pipe_id}/jobs/instances?jobType=Pipeline",
    json=body,
)
run_resp.raise_for_status()

run_id = run_resp.headers.get("x-ms-operation-id", "submitted")
print(f"\n✓ Pipeline triggered. Run ID: {run_id}")
notebookutils.notebook.exit(json.dumps({"triggered": True, "errors": len(errors_detected), "run_id": str(run_id)}))